# Laboratorio Semana 4 · Selección y filtrado crítico con Pandas
**Ciencia de datos con Python — SENA (material de apoyo GFPI-F-135)**  
**Caso:** seguimiento académico simulado. Continuidad: Semana 2 (carga e inspección) → Semana 3 (calidad y trazabilidad) → Semana 4 (subconjuntos e interpretación).

**Regla de trabajo:** utilice exclusivamente datos ficticios; no reporte cifras como hechos institucionales. El archivo fue **preparado**, no certificado. Conserve los casos conflictivos para revisión y distinga *registros* de *personas únicas*.


In [1]:
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FUENTE = RAIZ / "data" / "processed" / "seguimiento_academico_semana3_preparado.csv"
SALIDA = RAIZ / "outputs"
SALIDA.mkdir(parents=True, exist_ok=True)
assert FUENTE.exists(), f"No se encontró la fuente de la Semana 3: {FUENTE}"

df = pd.read_csv(
    FUENTE,
    dtype={"documento": "string", "ficha": "string"},
    parse_dates=["fecha_actualizacion"],
    encoding="utf-8-sig"
)
for campo in ["nota_python", "nota_datos", "asistencia"]:
    df[campo] = pd.to_numeric(df[campo], errors="coerce")
df["programa_normalizado"] = df["programa_normalizado"].astype("string")
df["ciudad_normalizada"] = df["ciudad_normalizada"].astype("string")
print("Pandas:", pd.__version__)
print("Filas y columnas:", df.shape)
print(df[["fila_fuente", "documento", "nombre", "programa_normalizado", "asistencia"]].head(5).to_string(index=False))

Pandas: 3.0.6
Filas y columnas: (19, 15)
 fila_fuente documento nombre programa_normalizado  asistencia
           2      1001    Ana                 ADSO        92.0
           3      1002 Carlos                 ADSO        85.0
           4      1003  María                 ADSO        96.0
           5      1004   Luis                 ADSO        70.0
           6      1005  Sofía                 ADSO        98.0


## 1. Recuperación de calidad: casos pendientes de la Semana 3
Una copia exacta fue retirada y quedó documentada. Sin embargo, **una clave de documento puede seguir repetida con calificaciones diferentes**. No decida cuál valor es verdadero: documente el conflicto. Los totales por filas no son automáticamente totales de personas.

In [2]:
repetido = df["documento"].notna() & df["documento"].duplicated(keep=False)
print("Total registros preparados:", len(df))
print("Registros con clave conflictiva:", int(repetido.sum()))
print(df.loc[repetido, ["fila_fuente", "documento", "nombre", "nota_python", "nota_datos"]].to_string(index=False))

# Exclusión SOLO del subconjunto de análisis que requiere identidad única:
base_unica = df.loc[df["documento"].notna() & ~repetido].copy()
print("Registros aptos según la regla didáctica de unicidad:", len(base_unica))

Total registros preparados: 19
Registros con clave conflictiva: 2
 fila_fuente documento nombre  nota_python  nota_datos
           8      1007 Camila          4.0         4.1
          21      1007 Camila          4.2         4.1
Registros aptos según la regla didáctica de unicidad: 17


## 2. Selección de columnas, loc e iloc
- `df["columna"]` devuelve Series; `df[["A", "B"]]` devuelve DataFrame.
- `.loc[filas, columnas]` selecciona etiquetas / condiciones[cite: 2].
- `.iloc[filas, columnas]` selecciona posiciones enteras; el extremo derecho del corte posicional es exclusivo[cite: 2]. No suponga que índices y posiciones son iguales[cite: 2].

In [3]:
print("Serie:", type(df["nombre"]).__name__)
print("Tabla:", type(df[["documento", "nombre"]]).__name__)
print("loc con columnas:")
print(df.loc[:, ["documento", "nombre", "asistencia"]].head(4).to_string(index=False))
print("iloc con cuatro primeras filas y tres primeras columnas:")
print(df.iloc[:4, :3].to_string(index=False))

# Selección de filas y columnas por condición:
print(df.loc[df["documento"].eq("1004"), ["documento", "nombre", "asistencia"]].to_string(index=False))

Serie: Series
Tabla: DataFrame
loc con columnas:
documento nombre  asistencia
     1001    Ana        92.0
     1002 Carlos        85.0
     1003  María        96.0
     1004   Luis        70.0
iloc con cuatro primeras filas y tres primeras columnas:
 fila_fuente documento nombre
           2      1001    Ana
           3      1002 Carlos
           4      1003  María
           5      1004   Luis
documento nombre  asistencia
     1004   Luis        70.0


## 3. Filtros booleanos y reglas explícitas
Conserve siempre los registros con datos desconocidos por separado. Un valor ausente no equivale a asistencia de cero y no debe considerarse bajo asistencia automáticamente[cite: 2].

In [4]:
umbral_asistencia = 80  # Regla DIDÁCTICA, no umbral institucional oficial
mascara_baja = df["asistencia"].notna() & (df["asistencia"] < umbral_asistencia)
baja_asistencia = df.loc[mascara_baja, ["documento", "nombre", "programa_normalizado", "asistencia"]].copy()
print("Registros bajo umbral:", len(baja_asistencia))
print(baja_asistencia.to_string(index=False))

pendientes_asistencia = df.loc[df["asistencia"].isna(), ["documento", "nombre", "asistencia"]]
print("Registros sin asistencia verificable:", len(pendientes_asistencia))

Registros bajo umbral: 2
documento nombre programa_normalizado  asistencia
     1004   Luis                 ADSO        70.0
     1008   Juan                 ADSO        75.0
Registros sin asistencia verificable: 3


## 4. isin() y between()
`isin()` busca categorías dentro de una colección; `between()` usa límites **incluyentes** por defecto[cite: 2]. Verifique explícitamente los valores ausentes antes de computar porcentajes[cite: 2].

In [5]:
programas = ["ADSO", "Ciencia de datos con Python"]
seleccion_programas = df.loc[df["programa_normalizado"].isin(programas),
    ["documento", "programa_normalizado", "asistencia"]]
print("Filas en programas definidos:", len(seleccion_programas))

rango_nota = df["nota_python"].notna() & df["nota_python"].between(3.0, 4.0, inclusive="both")
notas_intervalo = df.loc[rango_nota, ["documento", "nombre", "nota_python"]]
print("Registros con nota Python entre 3.0 y 4.0 (inclusive):", len(notas_intervalo))
print(notas_intervalo.to_string(index=False))

Filas en programas definidos: 19
Registros con nota Python entre 3.0 y 4.0 (inclusive): 7
documento nombre  nota_python
     1002 Carlos          3.8
     1006 Andrés          3.5
     1007 Camila          4.0
     1008   Juan          3.2
     1010  Diego          3.1
     1016  Mateo          3.6
     1017    NaN          4.0


## 5. Operadores lógicos `&`, `|` y `~`
Encierre **cada comparación** entre paréntesis[cite: 2]. `&` es Y; `|` es O; `~` invierte una condición[cite: 2]. Distinga los subconjuntos de registros en conflicto antes de emitir conclusiones[cite: 2].

In [6]:
# Regla 1: asistencia registrada menor de 80 Y programa ADSO
mascara_y = (
    df["programa_normalizado"].eq("ADSO")
    & df["asistencia"].notna()
    & df["asistencia"].lt(umbral_asistencia)
)
consulta_y = df.loc[mascara_y, ["documento", "nombre", "asistencia"]]
print("ADSO y asistencia menor de 80:", len(consulta_y))

# Regla 2: asistencia baja O nota Python conocida inferior a 3.5
mascara_o = (
    (df["asistencia"].notna() & df["asistencia"].lt(80))
    | (df["nota_python"].notna() & df["nota_python"].lt(3.5))
)
consulta_o = df.loc[mascara_o, ["documento", "nombre", "nota_python", "asistencia"]]
print("Coincide con al menos un criterio:", len(consulta_o))

# Regla 3: NO pertenecer al programa ADSO, solo entre programas informados
mascara_no = df["programa_normalizado"].notna() & ~(df["programa_normalizado"].eq("ADSO"))
consulta_no = df.loc[mascara_no, ["documento", "programa_normalizado"]]
print("Otros programas conocidos:", len(consulta_no))

ADSO y asistencia menor de 80: 2
Coincide con al menos un criterio: 3
Otros programas conocidos: 2


## 6. query() y variables externas
Use `@variable` para incorporar valores definidos en Python[cite: 2]. **Nunca ejecute cadenas de consulta construidas directamente con texto no confiable que provenga de usuarios[cite: 2].**

In [7]:
programa_objetivo = "ADSO"
minima = 80
consulta_query = df.query("programa_normalizado == @programa_objetivo and asistencia < @minima", engine="python")
consulta_booleana = df.loc[
    df["programa_normalizado"].eq(programa_objetivo)
    & df["asistencia"].notna() & df["asistencia"].lt(minima)
]
assert set(consulta_query.index) == set(consulta_booleana.index)
print("query y máscara producen los mismos índices:", len(consulta_query))

query y máscara producen los mismos índices: 2


## 7. Validación del subconjunto y lectura crítica
Formule el universo, justifique exclusiones, verifique condiciones y documente recuentos[cite: 2]. Si dos filas corresponden a un documento conflictivo, no afirme que representan dos aprendices[cite: 2].


In [8]:
base_sin_conflictos = df.loc[df["documento"].notna() & ~repetido].copy()
criterio = (
    base_sin_conflictos["programa_normalizado"].eq("ADSO")
    & base_sin_conflictos["asistencia"].notna()
    & base_sin_conflictos["asistencia"].lt(80)
)
subconjunto_validado = base_sin_conflictos.loc[criterio,
    ["fila_fuente", "documento", "nombre", "programa_normalizado", "asistencia", "archivo_fuente"]].copy()

assert subconjunto_validado["documento"].is_unique
assert subconjunto_validado["programa_normalizado"].eq("ADSO").all()
assert subconjunto_validado["asistencia"].lt(80).all()
assert subconjunto_validado["asistencia"].notna().all()
assert len(subconjunto_validado) == int(criterio.sum())
print(subconjunto_validado.to_string(index=False))
print("Entradas preparadas:",len(df),"| excluidas por conflicto:",int(repetido.sum()),
      "| base sin conflicto:",len(base_sin_conflictos), "| resultado:",len(subconjunto_validado))

registro_consulta = pd.DataFrame([{
    "consulta": "S4-01", "pregunta": "¿Qué registros únicos ADSO tienen asistencia <80?",
    "archivo_fuente": FUENTE.name, "umbral_didactico": 80,
    "registros_recibidos": len(df), "registros_con_clave_conflictiva": int(repetido.sum()),
    "registros_base": len(base_sin_conflictos), "registros_resultado": len(subconjunto_validado),
    "limitacion": "No corresponde a estadística oficial del SENA; datos ficticios."
}])

 fila_fuente documento nombre programa_normalizado  asistencia                                archivo_fuente
           5      1004   Luis                 ADSO        70.0 seguimiento_academico_semana3_con_errores.csv
           9      1008   Juan                 ADSO        75.0 seguimiento_academico_semana3_con_errores.csv
Entradas preparadas: 19 | excluidas por conflicto: 2 | base sin conflicto: 17 | resultado: 2


## 8. Guardar y comprobar evidencias
Exporte resultados a `outputs/`, no sobrescriba `data/raw` ni `data/processed`[cite: 2]. Contraste el CSV exportado con el DataFrame original, especialmente si hay tildes o identificadores[cite: 2].

In [9]:
salida_subconjunto = SALIDA / "semana4_adso_asistencia_baja_validado.csv"
salida_auditoria = SALIDA / "semana4_bitacora_consultas.csv"
subconjunto_validado.to_csv(salida_subconjunto,index=False,encoding="utf-8-sig")
registro_consulta.to_csv(salida_auditoria,index=False,encoding="utf-8-sig")
relectura = pd.read_csv(salida_subconjunto,dtype={"documento":"string"},encoding="utf-8-sig")
assert len(relectura)==len(subconjunto_validado)
assert relectura["documento"].tolist()==subconjunto_validado["documento"].tolist()
print("Archivos guardados:",salida_subconjunto.name,salida_auditoria.name)

Archivos guardados: semana4_adso_asistencia_baja_validado.csv semana4_bitacora_consultas.csv


## 9. Reto sin solución incluida (Desarrollo del Reto de Transferencia)
1. Seleccione registros del programa «Ciencia de datos con Python» cuya `nota_datos` sea válida y esté entre 3.0 y 4.5.
2. Formule una consulta que identifique registros con `ciudad_normalizada` faltante o `asistencia` faltante.
3. Compare dos filtros equivalentes, uno con máscara booleana y otro con `query()`.
4. Escriba conclusiones prudentes y limitaciones de la fuente.

In [10]:
# Reto 1: Ciencia de datos con Python y nota_datos entre 3.0 y 4.5
rango_datos = base_unica["nota_datos"].notna() & base_unica["nota_datos"].between(3.0, 4.5, inclusive="both")
mask_cd = base_unica["programa_normalizado"].eq("Ciencia de datos con Python") & rango_datos
sub_cd = base_unica.loc[mask_cd]
print("--- RETO 1 ---")
print(f"Entradas: {len(df)} | Excluidas por conflicto: {int(repetido.sum())} | Salida: {len(sub_cd)}")
print(sub_cd[["documento", "nombre", "programa_normalizado", "nota_datos"]].to_string(index=False))

# Reto 2: Ciudad normalizada faltante O asistencia faltante
mask_faltantes = base_unica["ciudad_normalizada"].isna() | base_unica["asistencia"].isna()
sub_faltantes = base_unica.loc[mask_faltantes]
print("\n--- RETO 2 ---")
print("Registros con ciudad o asistencia faltante:", len(sub_faltantes))

--- RETO 1 ---
Entradas: 19 | Excluidas por conflicto: 2 | Salida: 2
documento nombre        programa_normalizado  nota_datos
     1013  Pedro Ciencia de datos con Python         4.0
     1015  Erika Ciencia de datos con Python         3.0

--- RETO 2 ---
Registros con ciudad o asistencia faltante: 4
